# Cloud-Native Formats And Metadata

These notebooks show some of the conversion patterns used in the hackathon datasets. The focus is on file layout and metadata, not on deciding the scientifically correct reprojection or resampling method for every variable. Those choices remain dataset-specific.

The examples cover:

| Format | Best for | Why it matters in object storage |
| --- | --- | --- |
| Zarr | Multidimensional arrays such as time/depth/y/x cubes | Chunked reads let analysis load only the pieces it needs. |
| Cloud Optimized GeoTIFF (COG) | Single rasters or raster stacks exposed as files | Internal tiling and overviews enable efficient range reads and quicklooks. |
| GeoParquet | Vector geometries and tabular point data | Columnar storage, embedded geospatial metadata, and fast filtering. |


## Metadata

For example datatype we also show how we genereated its associated metadata. Rich metadata ensures that the datasets are more accessible - https://esa-earthcode.github.io/documentation/Community%20and%20Best%20Practices/FAIR%20and%20Open%20Science%20Best%20Practices/Data.


All the metadata generated for the datasets and hosted on the Open Science Catalog is in STAC format. The SpatioTemporal Asset Catalog (STAC) specification was designed to establish a standard, unified language to talk about geospatial data, allowing it to be more easily searchable and queryable. For more information and a introduction to STAC, see https://esa-earthcode.github.io/tutorials/stac-and-data-access/ . 

## Shared Local Data Directory

The conversion notebooks write examples under `downloaded_data/`, which is ignored by Git. The cell below creates the folder and records the small set of source files used by the format tutorials.


In [ ]:
from pathlib import Path


def find_repo_root(start: Path = Path.cwd()) -> Path:
    """Find the repository root when the kernel starts in a subfolder."""
    start = start.resolve()
    for path in (start, *start.parents):
        if (path / ".git").exists() or (path / "downloaded_data").exists():
            return path
    return start


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "downloaded_data"
DATA_DIR.mkdir(exist_ok=True)

print(f"Repository root: {REPO_ROOT}")
print(f"Data directory: {DATA_DIR}")


DOWNLOADS = {
    "ice_temperature_nc": {
        "url": "https://data.catds.fr/cecsm/Cryo_products/IceSheet_InternalTemperature/SM_TEST_MIR_ITUDP4_20130101T000000_20141231T000000_200_001_0.nc",
        "path": DATA_DIR / "SM_TEST_MIR_ITUDP4_20130101T000000_20141231T000000_200_001_0.nc",
    },
    "supraglacial_lakes_zip": {
        "url": "https://zenodo.org/api/records/5642755/files/WAIS_Max_Extent.zip/content",
        "path": DATA_DIR / "WAIS_Max_Extent.zip",
    },
    "basal_melt_geotiff": {
        "url": "https://s3.waw4-1.cloudferro.com/EarthCODE/OSCAssets/Polar_Ice_Shelves/antarctic-ice-shelf-melt-rates/basal_melt_map_racmo_firn_air_corrected.tif",
        "path": DATA_DIR / "basal_melt_map_racmo_firn_air_corrected.tif",
    },
    "albatros_tide_gauges_nc": {
        "url": "https://s3.waw4-1.cloudferro.com/EarthCODE/OSCAssets/polar_cube_datasets/albatross/datacollection_ALBATROSS_tidal_amplitude_phase_tide_gauges.nc",
        "path": DATA_DIR / "datacollection_ALBATROSS_tidal_amplitude_phase_tide_gauges.nc",
    },
}

In [ ]:
from urllib.request import urlretrieve

RUN_DOWNLOADS = False
OVERWRITE = False


def download_file(url: str, destination: Path, overwrite: bool = False) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and not overwrite:
        print(f"Already present: {destination}")
        return destination
    temporary = destination.with_suffix(destination.suffix + ".part")
    print(f"Downloading {url}\n  -> {destination}")
    urlretrieve(url, temporary)
    temporary.replace(destination)
    return destination


if RUN_DOWNLOADS:
    for item in DOWNLOADS.values():
        download_file(item["url"], item["path"], overwrite=OVERWRITE)
else:
    print("Set RUN_DOWNLOADS = True to download missing source files.")
